# Weather Prediction Results & Comparison

This notebook analyzes the performance of various weather prediction models. 

**Objective:** Identify the best model for precipitation forecasting across 1, 3, and 7-day horizons.

## Model Configurations

The following models were trained and evaluated. Note the difference in splitting strategies (Temporal vs. Sequential/Spatial).

### 1. Baselines (Reference)
* **Persistence:** Assumes tomorrow's weather is the same as today's. 
* **MA-3 (Moving Average):** Predicts using the average of the last 3 days.
* *Note:* We calculated these always within specific locations.

### 2. Tree Global (Gradient Boosting)
* **Library:** `sklearn.ensemble.GradientBoostingRegressor`
* **Training Split:** 80/20 Sequential Split (on concatenated data).
* **Key Params:** `n_estimators=300`, `learning_rate=0.05`, `max_depth=3`.
* **Features:** Lagged precipitation + Static attributes (Area, Elevation, Slope, Forest Fraction).

### 3. LightGBM Global
* **Library:** `lightgbm.LGBMRegressor`
* **Training Split:** 80/20 Sequential Split.
* **Key Params:** `n_estimators=500`, `learning_rate=0.05`, `num_leaves=31`.
* **Pros:** Significantly faster training than standard trees; uses `subsample=0.8` and `colsample_bytree=0.8` to prevent overfitting.

### 4. LSTM (Deep Learning)
* **Library:** `Darts` (BlockRNNModel)
* **Architecture:** RNN with LSTM cells.
* **Training Split:** **Temporal Split** (First 80% of *each* basin is Train, last 20% is Validation).
* **Input/Output:** Lookback window of **30 days** to predict **7 days** ahead.
* **Key Params:** `n_epochs=15`, `hidden_dim=20`, `n_rnn_layers=1`, `dropout=0.1`.
* **Loss Function:** L1 Loss (MAE) - less sensitive to outliers than MSE.

### 6. TFT (Temporal Fusion Transformer)
* **Library:** `Darts` (TFTModel)
* **Architecture:** Attention-based Transformer designed for multi-horizon forecasting.
* **Training Split:** Temporal Split (80/20).
* **Key Params:** `n_epochs=5`, `hidden_size=32`, `num_attention_heads=4`.
* **Pros:** Interpretable attention weights; handles static covariates natively.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")

RESULTS_DIR = Path("prediction_results/")

model_files = {
    "Baseline MA-3": "baseline/baseline_metrics_ma-3.csv",
    "Baseline Persistence": "baseline/baseline_metrics_persistence.csv",
    "Tree": "tree_global/metrics_tree_global.csv",
    "LightGBM": "lightgbm_global/metrics_lightgbm_global.csv",
    "LSTM": "lstm/nn_lstm_global_metrics_per_day.csv", 
    "TFT": "tft/tft_global_metrics_per_day.csv"
}
def normalize_horizon(target_name):
    t = target_name.lower()
    if '1d' in t: return '1 Day'
    if '3d' in t: return '3 Day'
    if '7d' in t: return '7 Day'
    return t

In [ ]:
dfs = []

for model_name, filename in model_files.items():
    file_path = RESULTS_DIR / filename
    
    if file_path.exists():
        temp_df = pd.read_csv(file_path)
        temp_df.columns = [c.strip() for c in temp_df.columns]
        
        # Handle Per-Location Files (Baselines)
        if 'location' in temp_df.columns:
            aggregated_rows = []
            for target in temp_df['target'].unique():
                subset = temp_df[temp_df['target'] == target]
                if 'n' in subset.columns:
                    total_n = subset['n'].sum()
                    global_rmse = np.sqrt((subset['RMSE']**2 * subset['n']).sum() / total_n)
                    global_mae = (subset['MAE'] * subset['n']).sum() / total_n
                else:
                    global_rmse = subset['RMSE'].mean()
                    global_mae = subset['MAE'].mean()
                
                aggregated_rows.append({
                    'target': target, 'model': model_name, 'RMSE': global_rmse, 'MAE': global_mae
                })
            temp_df = pd.DataFrame(aggregated_rows)

        # --- Standardize Columns ---
        temp_df['model'] = model_name
        temp_df['Horizon'] = temp_df['target'].apply(normalize_horizon)
        dfs.append(temp_df)
    else:
        print(f"[WARNING] File not found: {filename}")

if dfs:
    results_df = pd.concat(dfs, ignore_index=True)
    
    horizon_order = ["1 Day", "3 Day", "7 Day"]
    results_df['Horizon'] = pd.Categorical(results_df['Horizon'], categories=horizon_order, ordered=True)
    print("Data loaded successfully.")
else:
    print("No data loaded.")

In [ ]:
def plot_metric_sorted(df, metric="RMSE"):
    """
    Plots the given metric in 3 subplots (one per horizon).
    Sorts bars from Worst (High Error) to Best (Low Error).
    Highlights Baselines using user-defined colors.
    """
    horizons = ["1 Day", "3 Day", "7 Day"]
    
    # Increased figure height to accommodate larger text
    fig, axes = plt.subplots(1, 3, figsize=(20, 10), sharey=True)
    
    fig.suptitle(f"Model Comparison: {metric}", fontsize=24, weight='bold')
    
    for i, horizon in enumerate(horizons):
        ax = axes[i]
        
        data_h = df[df['Horizon'] == horizon].sort_values(by=metric, ascending=False)
        
        if data_h.empty: continue
            
        colors = []
        for model in data_h['model']:
            if "Baseline" in model:
                colors.append("#EDF5FA")
            else:
                colors.append("#9DC3E6")

        # Plot
        sns.barplot(
            data=data_h, 
            x="model", 
            y=metric, 
            hue="model", 
            ax=ax, 
            palette=colors,
            legend=False 
        )
        

        ax.set_title(f"Forecast Horizon: {horizon}", fontsize=18, weight='bold')
        ax.set_xlabel("")

        ax.tick_params(axis='x', rotation=45, labelsize=14)
        ax.tick_params(axis='y', labelsize=14)
        
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        
        if i == 0:
            ax.set_ylabel(f"{metric} (mm)", fontsize=16, weight='bold')
        else:
            ax.set_ylabel("")
            

        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f', padding=3, fontsize=14, weight='bold')

    plt.tight_layout()
    plt.subplots_adjust(top=0.88)
    plt.show()

In [ ]:
# ==========================================
# 4. Generate Plots
# ==========================================

if not results_df.empty:
    # 1. RMSE Plot
    plot_metric_sorted(results_df, metric="RMSE")
    
    # 2. MAE Plot
    plot_metric_sorted(results_df, metric="MAE")

In [ ]:
# ==========================================
# 5. Numeric Summary & Best Models
# ==========================================

if not results_df.empty:
    # --- A. Detailed Pivot Table ---
    print("\n=== Numeric Results (RMSE) ===")
    pivot_table = results_df.pivot(index='model', columns='Horizon', values='RMSE')
    
    # Sort table by the 1-Day performance roughly
    pivot_table = pivot_table.sort_values(by="1 Day", ascending=True)
    
    # Simple display without 'style' accessor to avoid Jinja2 errors
    display(pivot_table.round(4))
    
    # --- B. Explicit Text Analysis ---
    print("\n=== Performance Analysis ===")
    
    # Best Overall (Mean RMSE across all 3 horizons)
    overall_perf = results_df.groupby('model')['RMSE'].mean().sort_values()
    best_overall = overall_perf.index[0]
    
    print(f"🏆 BEST MODEL OVERALL: {best_overall} (Avg RMSE: {overall_perf.iloc[0]:.3f})")
    
    # Best per Horizon
    for horizon in ["1 Day", "3 Day", "7 Day"]:
        subset = results_df[results_df['Horizon'] == horizon]
        if not subset.empty:
            best_model = subset.loc[subset['RMSE'].idxmin()]
            print(f"   • Best for {horizon}: {best_model['model']} (RMSE: {best_model['RMSE']:.3f})")